# GPT from Scratch — Colab Training

**Before running:** `Runtime → Change runtime type → T4 GPU`

**Run cells top to bottom. Do not skip any cell.**

## 1. GPU check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## 2. Clone repo & set working directory

**Key:** We use `%cd` (IPython magic), not `os.chdir`.
`os.chdir` only affects the Python process — shell cells (`!cmd`) always
start from `/content` regardless. `%cd` changes the directory for both.

In [ ]:
!git clone https://github.com/DudeAj/gpt-from-scratch.git
%cd /content/gpt-from-scratch
!pwd
!ls

In [ ]:
# Install dependencies
!pip install tokenizers tqdm -q
print('Done.')

In [ ]:
!pip uninstall -y datasets huggingface_hub

!pip install \
    datasets==3.6.0 \
    huggingface_hub==0.34.4

## 3. Download datasets

Downloads WikiText-103 (~180 MB) and DailyDialog (~10 MB).  
Takes ~3 min. Skips files that already exist.

In [ ]:
!python -m dataset.download_data

# Verify files are in the right place
import os
for f in ['data/wikitext/train.txt', 'data/wikitext/validation.txt',
          'data/dailydialog/train.jsonl', 'data/dailydialog/validation.jsonl']:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1e6:.1f} MB' if exists else 'MISSING'
    print(f'  {f:<40} {size}')

## 4. Pretrain on WikiText-103

Expected on T4: **~45 min**  
First run tokenises the corpus (~5 min extra, cached after).  
Loss should drop from ~10 → ~4 over 3 epochs.

In [ ]:
!python -m training.pretrain

## 5. Fine-tune on DailyDialog

Expected on T4: **~15 min**  
Loss is masked to assistant turns only — lower numbers than pretrain.

In [ ]:
!python -m training.finetune

## 6. Test the model

In [ ]:
import torch, sys
sys.path.insert(0, '/content/gpt-from-scratch')

from model.gpt import GPT
from dataset.dialog_dataset import load_dialog_tokenizer
from inference.chat import generate_reply

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load('checkpoints/finetune_best.pt', map_location=device)
cfg  = ckpt['config']

tokenizer, _ = load_dialog_tokenizer()

model = GPT(
    vocab_size  = cfg['vocab_size'],
    max_seq_len = cfg['max_seq_len'],
    d_model     = cfg['d_model'],
    num_heads   = cfg['num_heads'],
    num_layers  = cfg['num_layers'],
    dropout     = cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model'])
print(f'Model loaded — epoch {ckpt["epoch"]}, val loss {ckpt["val_loss"]:.4f}\n')

prompts = [
    'Hello, how are you?',
    'What do you like to do on weekends?',
    'Can you recommend a good book?',
]

for p in prompts:
    reply = generate_reply(model, tokenizer, [p], temperature=0.8, top_k=40)
    print(f'You : {p}')
    print(f'Bot : {reply}')
    print()

## 7. Save checkpoints

**Option A — Download directly**

In [ ]:
!zip -r checkpoints.zip checkpoints/
from google.colab import files
files.download('checkpoints.zip')

**Option B — Save to Google Drive** (safer for large files or slow connections)

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copytree('checkpoints', '/content/drive/MyDrive/gpt-checkpoints', dirs_exist_ok=True)
print('Saved to Google Drive → MyDrive/gpt-checkpoints/')